In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv("data/lahore_aqi_historical_2024_onwards1.csv")

In [3]:
df.head(4)

,timestamp,temperature,humidity,wind_speed,pressure,modeled_pm25,modeled_pm10,ground_pm25,hour,day_of_week,month,aqi_change_rate,pm25_lag_1,pm25_lag_24,calculated_aqi,calculated_aqi_modeled
0,2024-01-01 00:00:00,6.3,98,2.8,992.3,155.1,222.6,141.436667,0,0,1,0.000000,NaN,NaN,195.0,206
1,2024-01-01 01:00:00,7.0,99,3.6,993.0,151.8,218.4,128.845000,1,0,1,-12.591667,141.436667,NaN,189.0,202
2,2024-01-01 02:00:00,7.2,98,3.2,993.5,153.7,220.7,131.483333,2,0,1,2.638333,128.845000,NaN,190.0,204
3,2024-01-01 03:00:00,7.9,99,2.8,994.2,156.0,224.4,140.615000,3,0,1,9.131667,131.483333,NaN,195.0,206


In [4]:
df.isnull().sum()

timestamp                    0
temperature                  0
humidity                     0
wind_speed                   0
pressure                     0
modeled_pm25                 0
modeled_pm10                 0
ground_pm25               7914
hour                         0
day_of_week                  0
month                        0
aqi_change_rate              0
pm25_lag_1                7915
pm25_lag_24               7938
calculated_aqi             108
calculated_aqi_modeled       0
dtype: int64

In [5]:
df.dropna(inplace=True)

In [7]:
df

,timestamp,temperature,humidity,wind_speed,pressure,modeled_pm25,modeled_pm10,ground_pm25,hour,day_of_week,month,aqi_change_rate,pm25_lag_1,pm25_lag_24,calculated_aqi,calculated_aqi_modeled
24,2024-01-02 00:00:00,5.7,99,2.6,991.2,188.1,270.1,169.805000,0,1,1,13.851667,155.953333,141.436667,220.0,238
25,2024-01-02 01:00:00,6.1,98,1.1,991.9,181.1,260.8,161.233333,1,1,1,-8.571667,169.805000,128.845000,212.0,231
26,2024-01-02 02:00:00,5.8,97,1.5,992.6,177.0,254.3,158.320000,2,1,1,-2.913333,161.233333,131.483333,209.0,227
27,2024-01-02 03:00:00,6.3,93,2.6,993.4,172.8,247.7,152.066667,3,1,1,-6.253333,158.320000,140.615000,203.0,223
28,2024-01-02 04:00:00,7.5,90,2.9,994.4,164.4,237.0,167.447500,4,1,1,15.380833,152.066667,142.643333,218.0,215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22651,2026-08-01 19:00:00,29.3,86,5.4,975.1,56.1,57.9,55.600000,19,5,8,-2.557143,58.157143,40.185714,151.0,151
22652,2026-08-01 20:00:00,29.1,85,6.7,975.3,54.4,56.1,45.714286,20,5,8,-9.885714,55.600000,44.342857,126.0,148
22653,2026-08-01 21:00:00,29.0,85,7.7,975.5,53.8,55.6,45.242857,21,5,8,-0.471429,45.714286,39.357143,125.0,146
22654,2026-08-01 22:00:00,28.7,87,6.8,975.6,54.5,56.1,45.328571,22,5,8,0.085714,45.242857,45.271429,125.0,148


In [10]:
print("Train date range:", df_model.iloc[:split_idx]['timestamp'].min() if 'timestamp' in df_model else 'timestamp not in feature set - check original df')
print("Test date range:", df_model.iloc[split_idx:]['timestamp'].min(), "to", df_model.iloc[split_idx:]['timestamp'].max())

Train date range: 2024-01-02 00:00:00
Test date range: 2026-04-14 18:00:00 to 2026-08-01 23:00:00


In [11]:
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor

LATITUDE = 31.5204
LONGITUDE = 74.3587
FORECAST_DAYS = 3

# ----------------------------------------------------
# AQI CALCULATION HELPERS (same as before)
# ----------------------------------------------------
PM25_BREAKPOINTS = [
    (0.0, 12.0, 0, 50), (12.1, 35.4, 51, 100), (35.5, 55.4, 101, 150),
    (55.5, 150.4, 151, 200), (150.5, 250.4, 201, 300),
    (250.5, 350.4, 301, 400), (350.5, 500.4, 401, 500),
]
PM10_BREAKPOINTS = [
    (0, 54, 0, 50), (55, 154, 51, 100), (155, 254, 101, 150),
    (255, 354, 151, 200), (355, 424, 201, 300),
    (425, 504, 301, 400), (505, 604, 401, 500),
]

def calculate_sub_index(concentration, breakpoints):
    if pd.isna(concentration) or concentration < 0:
        return None
    for c_lo, c_hi, i_lo, i_hi in breakpoints:
        if c_lo <= concentration <= c_hi:
            return round(((i_hi - i_lo) / (c_hi - c_lo)) * (concentration - c_lo) + i_lo)
    if concentration > breakpoints[-1][1]:
        return 500
    return None

def calculate_aqi(pm25, pm10):
    vals = [v for v in (calculate_sub_index(pm25, PM25_BREAKPOINTS),
                         calculate_sub_index(pm10, PM10_BREAKPOINTS)) if v is not None]
    return max(vals) if vals else None

def aqi_category(aqi):
    if aqi is None: return "Unknown"
    if aqi <= 50: return "Good"
    if aqi <= 100: return "Moderate"
    if aqi <= 150: return "Unhealthy for Sensitive Groups"
    if aqi <= 200: return "Unhealthy"
    if aqi <= 300: return "Very Unhealthy"
    return "Hazardous"

# ----------------------------------------------------
# STEP 1: Load historical data & train PM2.5 model
# ----------------------------------------------------
print("--- STEP 1: Training model on historical data ---")
df = pd.read_csv("data/lahore_aqi_historical_2024_onwards.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["pm25_roll_mean_3"] = df["ground_pm25"].rolling(3).mean().shift(1)
df["pm25_roll_mean_6"] = df["ground_pm25"].rolling(6).mean().shift(1)

feature_cols = [
    "temperature", "humidity", "wind_speed", "pressure",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "day_of_week",
    "pm25_lag_1", "pm25_lag_24", "pm25_roll_mean_3", "pm25_roll_mean_6",
]
target_col = "ground_pm25"

df_model = df.dropna(subset=feature_cols + [target_col])
X, y = df_model[feature_cols], df_model[target_col]

model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X, y)
print(f"Model trained on {len(X)} rows.")

# ----------------------------------------------------
# STEP 2: Fetch 3-day weather + pollutant forecasts
# ----------------------------------------------------
print("\n--- STEP 2: Fetching forecast data ---")

weather_res = requests.get("https://api.open-meteo.com/v1/forecast", params={
    "latitude": LATITUDE, "longitude": LONGITUDE,
    "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure",
    "forecast_days": FORECAST_DAYS, "timezone": "UTC"
})
weather_res.raise_for_status()
wd = weather_res.json()["hourly"]
df_weather_fc = pd.DataFrame({
    "timestamp": pd.to_datetime(wd["time"]),
    "temperature": wd["temperature_2m"],
    "humidity": wd["relative_humidity_2m"],
    "wind_speed": wd["wind_speed_10m"],
    "pressure": wd["surface_pressure"],
})

aq_res = requests.get("https://air-quality-api.open-meteo.com/v1/air-quality", params={
    "latitude": LATITUDE, "longitude": LONGITUDE,
    "hourly": "pm2_5,pm10",
    "forecast_days": FORECAST_DAYS, "timezone": "UTC"
})
aq_res.raise_for_status()
aqd = aq_res.json()["hourly"]
df_aq_fc = pd.DataFrame({
    "timestamp": pd.to_datetime(aqd["time"]),
    "modeled_pm25": aqd["pm2_5"],
    "modeled_pm10": aqd["pm10"],
})

df_forecast = pd.merge(df_weather_fc, df_aq_fc, on="timestamp", how="inner")
df_forecast = df_forecast[df_forecast["timestamp"] > df["timestamp"].max()].reset_index(drop=True)
print(f"Forecast horizon: {len(df_forecast)} hours ({df_forecast['timestamp'].min()} to {df_forecast['timestamp'].max()})")

# ----------------------------------------------------
# STEP 3: Recursive hourly forecasting
# ----------------------------------------------------
print("\n--- STEP 3: Running recursive forecast ---")

# Seed recent history (last 24+ hrs of real ground_pm25) for lag/rolling calc
history = df[["timestamp", target_col]].dropna().set_index("timestamp")[target_col].to_dict()

results = []
for _, row in df_forecast.iterrows():
    ts = row["timestamp"]

    lag_1_ts = ts - pd.Timedelta(hours=1)
    lag_24_ts = ts - pd.Timedelta(hours=24)
    pm25_lag_1 = history.get(lag_1_ts, np.nan)
    pm25_lag_24 = history.get(lag_24_ts, np.nan)

    roll_vals_3 = [history.get(ts - pd.Timedelta(hours=h), np.nan) for h in range(1, 4)]
    roll_vals_6 = [history.get(ts - pd.Timedelta(hours=h), np.nan) for h in range(1, 7)]
    roll_mean_3 = np.nanmean(roll_vals_3) if not all(pd.isna(roll_vals_3)) else pm25_lag_1
    roll_mean_6 = np.nanmean(roll_vals_6) if not all(pd.isna(roll_vals_6)) else pm25_lag_1

    features = pd.DataFrame([{
        "temperature": row["temperature"],
        "humidity": row["humidity"],
        "wind_speed": row["wind_speed"],
        "pressure": row["pressure"],
        "hour_sin": np.sin(2 * np.pi * ts.hour / 24),
        "hour_cos": np.cos(2 * np.pi * ts.hour / 24),
        "month_sin": np.sin(2 * np.pi * ts.month / 12),
        "month_cos": np.cos(2 * np.pi * ts.month / 12),
        "day_of_week": ts.dayofweek,
        "pm25_lag_1": pm25_lag_1,
        "pm25_lag_24": pm25_lag_24,
        "pm25_roll_mean_3": roll_mean_3,
        "pm25_roll_mean_6": roll_mean_6,
    }])[feature_cols]

    pred_pm25 = model.predict(features)[0]
    history[ts] = pred_pm25  # feed forward for next iteration's lags

    pred_aqi = calculate_aqi(pred_pm25, row["modeled_pm10"])

    results.append({
        "timestamp": ts,
        "predicted_pm25": round(pred_pm25, 1),
        "predicted_aqi": pred_aqi,
        "category": aqi_category(pred_aqi),
    })

df_result = pd.DataFrame(results)

# ----------------------------------------------------
# STEP 4: Save + summarize
# ----------------------------------------------------
df_result.to_csv("data/lahore_aqi_3day_forecast.csv", index=False)

df_result["date"] = df_result["timestamp"].dt.date
daily_summary = df_result.groupby("date").agg(
    avg_aqi=("predicted_aqi", "mean"),
    max_aqi=("predicted_aqi", "max"),
    min_aqi=("predicted_aqi", "min"),
).round(1)

print("\n--- 3-Day Daily AQI Summary ---")
print(daily_summary)
print(f"\nFull hourly forecast saved to: data/lahore_aqi_3day_forecast.csv")

--- STEP 1: Training model on historical data ---
Model trained on 12684 rows.

--- STEP 2: Fetching forecast data ---
Forecast horizon: 72 hours (2026-08-02 00:00:00 to 2026-08-04 23:00:00)

--- STEP 3: Running recursive forecast ---

--- 3-Day Daily AQI Summary ---
            avg_aqi  max_aqi  min_aqi
date                                 
2026-08-02    131.3      150      117
2026-08-03    134.7      158      111
2026-08-04    142.0      161      111

Full hourly forecast saved to: data/lahore_aqi_3day_forecast.csv
